# Assignment 3

## Data Cleaning and Transformation Steps

In [1]:
import pandas as pd
import ast

### Import raw data

Loaded the US_inflation_rates.csv and full_movie_data.csv files into pandas DataFrames.

In [2]:
inflation_df = pd.read_csv('./data/ready/US_inflation_rates_ready.csv')
movies_df = pd.read_csv('./data/raw/full_movie_data.csv')

In [3]:
inflation_df.head()

,date,value,Is_Recession
0,1947,22.331667,No
1,1948,24.045000,False
2,1949,23.809167,TRUE
3,1950,24.062500,No
4,1951,25.973333,FALSE


In [4]:
movies_df.head()

,Title,Year,Rated,Released,Runtime,Genre,Director,Writer,Actors,Plot,...,Metascore,imdbRating,imdbVotes,imdbID,Type,DVD,BoxOffice,Production,Website,Response
0,The Shawshank Redemption,1994,R,14 Oct 1994,142 min,Drama,Frank Darabont,"Stephen King, Frank Darabont","Tim Robbins, Morgan Freeman, Bob Gunton",A banker convicted of uxoricide forms a friend...,...,82.0,9.3,"3,136,948",tt0111161,movie,NaN,"$28,767,189",NaN,NaN,True
1,The Dark Knight,2008,PG-13,18 Jul 2008,152 min,"Action, Crime, Drama",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Goyer","Christian Bale, Heath Ledger, Aaron Eckhart",When a menace known as the Joker wreaks havoc ...,...,85.0,9.1,"3,115,102",tt0468569,movie,NaN,"$534,987,076",NaN,NaN,True
2,Inception,2010,PG-13,16 Jul 2010,148 min,"Action, Adventure, Sci-Fi",Christopher Nolan,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...",A thief who steals corporate secrets through t...,...,74.0,8.8,"2,767,518",tt1375666,movie,NaN,"$292,587,330",NaN,NaN,True
3,Fight Club,1999,R,15 Oct 1999,139 min,"Crime, Drama, Thriller",David Fincher,"Chuck Palahniuk, Jim Uhls","Brad Pitt, Edward Norton, Meat Loaf",An insomniac office worker and a devil-may-car...,...,67.0,8.8,"2,552,332",tt0137523,movie,NaN,"$37,030,102",NaN,NaN,True
4,Interstellar,2014,PG-13,07 Nov 2014,169 min,"Adventure, Drama, Sci-Fi",Christopher Nolan,"Jonathan Nolan, Christopher Nolan","Matthew McConaughey, Anne Hathaway, Jessica Ch...",When Earth becomes uninhabitable in the future...,...,74.0,8.7,"2,454,660",tt0816692,movie,NaN,"$203,227,580",NaN,NaN,True


### Clean inflation data and merge it with the Movie dataset

Renamed the `date` column to `Year` and `value` to `CPI`.

Standardized the `Is_Recession` column to Boolean values

Merged the movie data with the inflation data based on the `Year` columns.

In [5]:
inflation_df = inflation_df.rename(columns={'date': 'Year','value': 'CPI'})

def clean_recession(x):
    x_str = str(x).lower()
    if x_str in ['1', 'yes', 'true']:
        return True
    elif x_str in ['0', 'no', 'false']:
        return False
    return None

inflation_df['Is_Recession'] = inflation_df['Is_Recession'].apply(clean_recession)

inflation_df.head()

,Year,CPI,Is_Recession
0,1947,22.331667,False
1,1948,24.045000,False
2,1949,23.809167,True
3,1950,24.062500,False
4,1951,25.973333,False


In [6]:
merged_df = pd.merge(movies_df, inflation_df, on='Year', how='left')
merged_df.head()

,Title,Year,Rated,Released,Runtime,Genre,Director,Writer,Actors,Plot,...,imdbVotes,imdbID,Type,DVD,BoxOffice,Production,Website,Response,CPI,Is_Recession
0,The Shawshank Redemption,1994,R,14 Oct 1994,142 min,Drama,Frank Darabont,"Stephen King, Frank Darabont","Tim Robbins, Morgan Freeman, Bob Gunton",A banker convicted of uxoricide forms a friend...,...,"3,136,948",tt0111161,movie,NaN,"$28,767,189",NaN,NaN,True,148.225000,False
1,The Dark Knight,2008,PG-13,18 Jul 2008,152 min,"Action, Crime, Drama",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Goyer","Christian Bale, Heath Ledger, Aaron Eckhart",When a menace known as the Joker wreaks havoc ...,...,"3,115,102",tt0468569,movie,NaN,"$534,987,076",NaN,NaN,True,215.254250,True
2,Inception,2010,PG-13,16 Jul 2010,148 min,"Action, Adventure, Sci-Fi",Christopher Nolan,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...",A thief who steals corporate secrets through t...,...,"2,767,518",tt1375666,movie,NaN,"$292,587,330",NaN,NaN,True,218.076167,False
3,Fight Club,1999,R,15 Oct 1999,139 min,"Crime, Drama, Thriller",David Fincher,"Chuck Palahniuk, Jim Uhls","Brad Pitt, Edward Norton, Meat Loaf",An insomniac office worker and a devil-may-car...,...,"2,552,332",tt0137523,movie,NaN,"$37,030,102",NaN,NaN,True,166.583333,False
4,Interstellar,2014,PG-13,07 Nov 2014,169 min,"Adventure, Drama, Sci-Fi",Christopher Nolan,"Jonathan Nolan, Christopher Nolan","Matthew McConaughey, Anne Hathaway, Jessica Ch...",When Earth becomes uninhabitable in the future...,...,"2,454,660",tt0816692,movie,NaN,"$203,227,580",NaN,NaN,True,236.715000,False


### Clean the dataset

Removed non-numeric characters ($ and ,) for `BoxOffice` columns.

Created a new attribute, `AdjustedBoxOffice`, to normalize financial success across eras. This was calculated using the formula:

`AdjustedBoxOffice = BoxOffice * ( Latest CPI / CPI at Release )`

In [7]:
current_cpi = inflation_df.loc[inflation_df['Year'] == 2025, 'CPI'].values[0]

merged_df['BoxOffice'] = merged_df['BoxOffice'].astype(str).str.replace('$', '', regex=False).str.replace(',', '')

merged_df['AdjustedBoxOffice'] = (merged_df['BoxOffice'].astype(float) * (current_cpi / merged_df['CPI'])).round(0).astype('Int64')

merged_df[['BoxOffice','AdjustedBoxOffice']].head()

,BoxOffice,AdjustedBoxOffice
0,28767189,62662497
1,534987076,802459799
2,292587330,433190616
3,37030102,71772007
4,203227580,277197045


Parsed the `Ratings` column to extract `Rotten Tomatoes` score, since it was the only rating that didn't have its own column.

Converted Metascore column to integer format.

In [8]:
def extract_rotten_tomatoes(rating):
    try:
        rating_list = ast.literal_eval(rating) if isinstance(rating, str) else rating
        if isinstance(rating_list, list):
            for source in rating_list:
                if source.get('Source') == 'Rotten Tomatoes':
                    value = source.get('Value')
                    if isinstance(value, str) and value.endswith('%'):
                        return int(value.rstrip('%'))
    except Exception:
        pass
    return None

merged_df['RottenTomatoes'] = merged_df['Ratings'].apply(extract_rotten_tomatoes).astype('Int64')
merged_df['Metascore'] = merged_df['Metascore'].astype('Int64')
merged_df[['RottenTomatoes', 'imdbRating', 'Metascore']].head()

,RottenTomatoes,imdbRating,Metascore
0,89,9.3,82
1,94,9.1,85
2,87,8.8,74
3,81,8.8,67
4,73,8.7,74


Dropped columns I thought were unnecessary for this project.

Dropped rows that were missing values.

In [9]:
merged_df = merged_df.drop(columns=['Released','Runtime','Director','Writer','Actors','Rated','DVD', 'Production', 'Website', 'Response','Language','Country','Awards','Plot','Poster','imdbVotes','Ratings','imdbID','Type','BoxOffice', 'CPI'])
merged_df = merged_df.dropna()

In [10]:
merged_df.head()

,Title,Year,Genre,Metascore,imdbRating,Is_Recession,AdjustedBoxOffice,RottenTomatoes
0,The Shawshank Redemption,1994,Drama,82,9.3,False,62662497,89
1,The Dark Knight,2008,"Action, Crime, Drama",85,9.1,True,802459799,94
2,Inception,2010,"Action, Adventure, Sci-Fi",74,8.8,False,433190616,87
3,Fight Club,1999,"Crime, Drama, Thriller",67,8.8,False,71772007,81
4,Interstellar,2014,"Adventure, Drama, Sci-Fi",74,8.7,False,277197045,73


Saved as CSV file.

In [11]:
merged_df.to_csv('./data/ready/movies_with_inflation_data_ready.csv', index=False)